# LSTM — predicao de falha em SSDs (7 dias)

Notebook de treino do LSTM, adaptado para rodar **localmente** ou no **Colab**
(deteccao automatica na celula de ambiente). Treina por epocas, **guarda o melhor
modelo** (menor perda de validacao) e tambem o **historico completo de todas as
epocas** — para os graficos saírem completos (mostrando inclusive o que acontece
depois da melhor epoca). Salva o checkpoint em `OUTPUT_DIR` no mesmo formato que
`AMA_comparacao_modelos` consome.

> No Colab, selecione **GPU** (Ambiente de execucao -> Alterar tipo de ambiente
> -> T4 GPU). O treino completo e demorado; para um teste rapido reduza `EPOCHS`
> e/ou `actual_patience` na celula de treino.

In [ ]:
# ===== Configuracao de ambiente (execucao LOCAL) =====
import os, sys

def _encontra_raiz(inicio=None):
    d = os.path.abspath(inicio or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, 'comum', 'ssd_utils.py')):
            return d
        pai = os.path.dirname(d)
        if pai == d:
            raise RuntimeError('raiz do projeto (com comum/ssd_utils.py) nao encontrada')
        d = pai

PROJ_ROOT = _encontra_raiz()
COMUM_DIR = os.path.join(PROJ_ROOT, 'comum')
H7_DIR = os.path.join(PROJ_ROOT, 'horizonte_7dias')
H30_DIR = os.path.join(PROJ_ROOT, 'horizonte_30dias')
DATA_DIR = COMUM_DIR                 # data.pickle / mask.pickle
OUTPUT_DIR = H7_DIR            # onde salvar o checkpoint e as figuras
CONTAMINATION_LEVEL = 7         # horizonte de previsao de falha
CHECKPOINT_NAME = 'lstm_7_dias.pth'
FIG_SUFFIX = ''
HORIZ_LABEL = '7 dias'
print('Execucao LOCAL | PROJ =', PROJ_ROOT, '| OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import pickle
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import time
import math
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print(device)

Using GPU: Tesla T4


In [ ]:
# (caminhos definidos na celula de ambiente: DATA_DIR / OUTPUT_DIR)

In [ ]:
with open(os.path.join(DATA_DIR, 'data.pickle'), 'rb') as f:
    all_data = pickle.load(f)

with open(os.path.join(DATA_DIR, 'mask.pickle'), 'rb') as f:
    all_mask = pickle.load(f)

print('data:', all_data.shape, '| mask:', all_mask.shape)


In [ ]:
train_data = all_data[0:3740].copy()
train_mask = all_mask[0:3740].copy()

In [ ]:
test_data = all_data[3740:4541].copy()
test_mask = all_mask[3740:4541].copy()

In [ ]:
validation_data = all_data[4541:].copy()
validation_mask = all_mask[4541:].copy()

In [ ]:
def create_class_labels(data, mask, contamination_level):
    class_labels = []

    for i in range(len(data)):
        zero_list = np.zeros(360)

        try:
            first_zero = np.where(mask[i] == 0)[0][0]
        except:
            first_zero = 360

        try:
            zero_list[first_zero - contamination_level:first_zero] = [1] * contamination_level
        except:
            # ssds com tamanho menor que o nivel de contaminação
            zero_list[first_zero - first_zero:first_zero] = [1] * (first_zero)

        class_labels.append(zero_list)

    return class_labels

In [ ]:
train_labels = create_class_labels(train_data, train_mask, CONTAMINATION_LEVEL)
test_labels = create_class_labels(test_data, test_mask, CONTAMINATION_LEVEL)
validation_labels = create_class_labels(validation_data, validation_mask, CONTAMINATION_LEVEL)

In [ ]:
def get_valid_size(mask):
    valid_size = []
    for i in range(len(mask)):
        valid_size.append(mask[i].sum())
    return np.asarray(valid_size)

In [ ]:
train_valid_sizes = get_valid_size(train_mask)
test_valid_sizes = get_valid_size(test_mask)
validation_valid_sizes = get_valid_size(validation_mask)

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y, mask, valid_sizes):
        self.X = X
        self.y = y
        self.mask = mask
        self.valid_sizes = valid_sizes

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.mask[idx], self.valid_sizes[idx]

In [ ]:
train_dataset = SequenceDataset(train_data, train_labels, train_mask, train_valid_sizes)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

test_dataset = SequenceDataset(test_data, test_labels, test_mask, test_valid_sizes)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

validation_dataset = SequenceDataset(validation_data, validation_labels, validation_mask, validation_valid_sizes)
validation_loader = DataLoader(validation_dataset, batch_size=128, shuffle=False)

In [ ]:
def batch_evaluation(loader, model, loss_fn):
    model.eval()

    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (padded, label, mask, valid_sizes) in enumerate(loader):
            padded = padded.to(device)
            label = label.to(device)
            mask = mask.to(device)
            #no need to put valid_sizes in GPU

            output = model(padded.float(), valid_sizes)
            output = output.squeeze()

            loss = loss_fn(output, label)
            loss = loss * mask
            loss = loss.sum() / mask.sum()

            total_loss += loss.item()

            probs = torch.sigmoid(output)
            preds = (probs > 0.5).float()

            preds = preds[mask == 1]
            valid_labels = label[mask == 1]

            all_preds.append(preds.cpu())
            all_labels.append(valid_labels.cpu())

    epoch_loss = total_loss / len(loader)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    accuracy = (all_preds == all_labels).mean()
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    if precision + recall == 0:
        f1_score = 0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)
    cm = confusion_matrix(all_labels, all_preds)

    return total_loss, epoch_loss, accuracy, precision, recall, f1_score, cm


In [ ]:
import copy
start_time = time.time()  # seconds

class LSTM(torch.nn.Module):
    def __init__(self):
        super(LSTM, self).__init__()
        self.lstm_layers = torch.nn.ModuleList()
        self.lstm_layers.append(torch.nn.LSTM(num_layers=1, input_size=24, hidden_size=256, batch_first=True))
        self.lstm_layers.append(torch.nn.LSTM(num_layers=1, input_size=256, hidden_size=128, batch_first=True))
        self.lstm_layers.append(torch.nn.LSTM(num_layers=1, input_size=128, hidden_size=64, batch_first=True))
        self.linear1 = torch.nn.Linear(64, 32)
        self.linear2 = torch.nn.Linear(32, 16)
        self.linear3 = torch.nn.Linear(16, 1)
        self.relu = torch.nn.ReLU()

    def forward(self, padded, lengths):
        packed = pack_padded_sequence(padded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out = packed
        for lstm in self.lstm_layers:
            out, _ = lstm(out)
        x, _ = pad_packed_sequence(out, batch_first=True)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)
        return x

lstm = LSTM().to(device)
loss_fn = torch.nn.BCEWithLogitsLoss(reduction='none').to(device)
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.001)

EPOCHS = 10000
PATIENCE = 1000          # epocas sem melhora ate parar (reduza para testes rapidos)

total_train_loss_list = []
total_test_loss_list = []
total_validation_loss_list = []
# metricas por epoca (para graficos completos)
train_f1_list = []
test_f1_list = []
val_f1_list = []
val_precision_list = []
val_recall_list = []

actual_patience = PATIENCE
best_validation_loss = math.inf
best_model_epoch = 0
best_state = None
best_optimizer_state = None
best_time = 0.0

for epoch in range(EPOCHS):
    lstm.train()
    total_train_loss = 0
    all_train_preds = []
    all_train_labels = []

    for i, (padded, label, mask, valid_sizes) in enumerate(train_loader):
        padded = padded.to(device)
        label = label.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()
        output = lstm(padded.float(), valid_sizes)
        output = output.squeeze()

        loss = loss_fn(output, label)
        loss = loss * mask
        loss = loss.sum() / mask.sum()

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        probs = torch.sigmoid(output)
        preds = (probs > 0.5).float()
        preds = preds[mask == 1]
        valid_labels = label[mask == 1]
        all_train_preds.append(preds.cpu())
        all_train_labels.append(valid_labels.cpu())

    epoch_loss_train = total_train_loss / len(train_loader)
    all_train_preds = torch.cat(all_train_preds).numpy()
    all_train_labels = torch.cat(all_train_labels).numpy()
    accuracy_train = (all_train_preds == all_train_labels).mean()
    precision_train = precision_score(all_train_labels, all_train_preds, zero_division=0)
    recall_train = recall_score(all_train_labels, all_train_preds, zero_division=0)
    if precision_train + recall_train == 0:
        f1_score_train = 0
    else:
        f1_score_train = 2 * (precision_train * recall_train) / (precision_train + recall_train)
    cm_train = confusion_matrix(all_train_labels, all_train_preds)

    (test_loss, test_epoch_loss, accuracy_test, precision_test,
     recall_test, f1_score_test, cm_test) = batch_evaluation(test_loader, lstm, loss_fn)

    (validation_loss, validation_epoch_loss, accuracy_validation, precision_validation,
     recall_validation, f1_score_validation, cm_validation) = batch_evaluation(validation_loader, lstm, loss_fn)

    # historico COMPLETO (todas as epocas)
    total_train_loss_list.append(epoch_loss_train)
    total_test_loss_list.append(test_epoch_loss)
    total_validation_loss_list.append(validation_epoch_loss)
    train_f1_list.append(f1_score_train)
    test_f1_list.append(f1_score_test)
    val_f1_list.append(f1_score_validation)
    val_precision_list.append(precision_validation)
    val_recall_list.append(recall_validation)

    time_elapsed_in_seconds = time.time() - start_time

    if epoch % 10 == 0 or validation_epoch_loss < best_validation_loss:
        print(f"Epoch: {epoch} Time Elapsed: {time_elapsed_in_seconds:.2f}s")
        print(f"TREINO      loss={epoch_loss_train:.6f}  acc={accuracy_train:.6f}  prec={precision_train:.6f}  rec={recall_train:.6f}  f1={f1_score_train:.6f}")
        print(f"TESTE       loss={test_epoch_loss:.6f}  acc={accuracy_test:.6f}  prec={precision_test:.6f}  rec={recall_test:.6f}  f1={f1_score_test:.6f}")
        print(f"VALIDACAO   loss={validation_epoch_loss:.6f}  acc={accuracy_validation:.6f}  prec={precision_validation:.6f}  rec={recall_validation:.6f}  f1={f1_score_validation:.6f}")

    # guarda o MELHOR modelo em memoria (nao para o treino: deixa o historico continuar)
    if validation_epoch_loss < best_validation_loss:
        best_validation_loss = validation_epoch_loss
        best_model_epoch = epoch + 1
        best_state = copy.deepcopy(lstm.state_dict())
        best_optimizer_state = copy.deepcopy(optimizer.state_dict())
        best_time = time_elapsed_in_seconds
        actual_patience = PATIENCE
    else:
        actual_patience -= 1

    if actual_patience == 0:
        print(f"Early stopping na epoca {epoch+1} (sem melhora ha {PATIENCE} epocas).")
        break

# ----- Salva UMA vez: melhor modelo + historico COMPLETO de todas as epocas treinadas -----
total_time_elapsed = time.time() - start_time
checkpoint = {
    'best_epoch': best_model_epoch,
    'best_model_state_dict': best_state,
    'optimizer_state_dict': best_optimizer_state,
    'best_validation_loss': best_validation_loss,
    'total_train_loss_list': total_train_loss_list,
    'total_test_loss_list': total_test_loss_list,
    'total_validation_loss_list': total_validation_loss_list,
    'val_f1_list': val_f1_list,
    'val_precision_list': val_precision_list,
    'val_recall_list': val_recall_list,
    'test_f1_list': test_f1_list,
    'train_f1_list': train_f1_list,
    'time_elapsed': best_time,                 # tempo ate a melhor epoca (compat. com a comparacao)
    'total_time_elapsed': total_time_elapsed,  # tempo total de treino
    'epochs_trained': len(total_train_loss_list),
    'device': str(device),
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.save(checkpoint, os.path.join(OUTPUT_DIR, CHECKPOINT_NAME))
print(f"\nTreino concluido: {len(total_train_loss_list)} epocas | melhor epoca: {best_model_epoch} "
      f"| melhor val loss: {best_validation_loss:.6f}")
print(f"Checkpoint salvo em {os.path.join(OUTPUT_DIR, CHECKPOINT_NAME)}")


## Curva de perda por epoca (completa)

Perda BCE de treino/teste/validacao em **todas as epocas treinadas** (nao para na
melhor epoca) - a linha tracejada marca a melhor epoca (a usada pelo modelo
salvo). Assim da para ver a convergencia e o eventual overfitting depois do
melhor ponto.

In [ ]:
import matplotlib.pyplot as plt

epcs = range(1, len(total_train_loss_list) + 1)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epcs, total_train_loss_list, label='Treino', lw=1.4)
ax.plot(epcs, total_test_loss_list, label='Teste', lw=1.4)
ax.plot(epcs, total_validation_loss_list, label='Validacao', lw=1.4)
ax.axvline(best_model_epoch, color='gray', ls='--', lw=1.2, label=f'Melhor epoca ({best_model_epoch})')
ax.set_xlabel('Epoca')
ax.set_ylabel('BCE Loss (media por batch)')
ax.set_title(f'LSTM - historico completo de perda ({HORIZ_LABEL})')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, f'lstm_train_loss{FIG_SUFFIX}.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em', f'lstm_train_loss{FIG_SUFFIX}.png')


## Metricas de validacao por epoca

Precisao, recall e F1 da classe "vai falhar" no conjunto de validacao, epoca a
epoca. Util para ver a evolucao da capacidade de deteccao ao longo do treino
(analogo a "metrica melhorando por epoca").

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epcs, val_precision_list, label='Precisao (val)', lw=1.4)
ax.plot(epcs, val_recall_list, label='Recall (val)', lw=1.4)
ax.plot(epcs, val_f1_list, label='F1 (val)', lw=1.6)
ax.axvline(best_model_epoch, color='gray', ls='--', lw=1.2, label=f'Melhor epoca ({best_model_epoch})')
ax.set_xlabel('Epoca')
ax.set_ylabel('Metrica (validacao)')
ax.set_title(f'LSTM - metricas de validacao por epoca ({HORIZ_LABEL})')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, f'lstm_train_metrics{FIG_SUFFIX}.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em', f'lstm_train_metrics{FIG_SUFFIX}.png')
